# uav-traffic-vision — YOLO26s-obb on DOTA-v1.0 (Colab training, Phase 3)

One-click training notebook: **Runtime → Run all**.

Phase 3 supplement to the main VisDrone track: same aerial-imagery domain, a different
annotation format — **oriented bounding boxes (OBB)** instead of axis-aligned boxes.

What it does:
1. Downloads DOTA-v1.0 (~2 GB, 1,411 train / 458 val images, 15 classes) and tiles it
   into overlapping 1024x1024 crops with `split_dota` — DOTA images run up to ~13,000 px
   on a side and can't be fed to YOLO directly (see `reports/dota_stats.md` for the
   measured size distribution; the local repo's tiling run produced 15,749 train /
   5,297 val tiles from the 1,869 raw images).
2. Trains `yolo26s-obb` at **imgsz=1024** (the tile size — no separate resolution
   comparison here, unlike the VisDrone track).
3. Checkpoints land directly on **Google Drive** (`MyDrive/uav-traffic-vision/runs/...`),
   so a disconnect never loses progress — see the resume note below.
4. Validates `best.pt` (built-in rotated-IoU OBB mAP) and copies it to a stable Drive path.

Requirements: Colab Pro, GPU runtime (**L4** or **A100** — Runtime → Change runtime type).

In [ ]:
# GPU / CPU check
import os
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"CPU cores: {os.cpu_count()}")

In [ ]:
# Mount Google Drive (checkpoints and final weights live here)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Configuration
from pathlib import Path

MODEL    = "yolo26s-obb.pt"
IMGSZ    = 1024  # = split_dota's tile size; DOTA's native training resolution
EPOCHS   = 100
PATIENCE = 20

# batch tuned conservatively: OBB's per-tile memory footprint at imgsz=1024 hadn't
# been measured on either GPU before this notebook existed, and the VisDrone 1024
# experiment already showed A100 80GB hitting 93% VRAM (and still climbing) at
# batch=32 for the lighter plain-detect head -- OBB adds an angle-regression branch
# on top, so start conservative and raise only after checking GPU_mem in the first
# epoch's log.
if "A100" in GPU_NAME:
    BATCH = 24
elif "L4" in GPU_NAME:
    BATCH = 8
else:
    BATCH = 8  # conservative default for an unrecognized GPU

CACHE    = "disk"  # cache decoded tiles on local disk (same win as the VisDrone notebook)
SEED     = 42
RESUME   = False   # after a disconnect: set True, Runtime -> Run all again

DRIVE_ROOT  = Path("/content/drive/MyDrive/uav-traffic-vision")
RUNS_DIR    = DRIVE_ROOT / "runs"     # ultralytics project dir -> checkpoints on Drive
WEIGHTS_DIR = DRIVE_ROOT / "weights"  # stable copies of best.pt / results.csv
RUN_NAME    = f"dota_obb_yolo26s_{IMGSZ}"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU={GPU_NAME}  BATCH={BATCH}  cache={CACHE}")

In [ ]:
# Install ultralytics + shapely (split_dota's bbox_iof needs it), pinned to the
# versions locked in the repo's uv.lock
%pip install -q ultralytics==8.4.90 shapely==2.1.2

from ultralytics import settings
settings.update({"datasets_dir": "/content/datasets"})  # Colab local disk, NOT Drive (fast I/O)

In [ ]:
# Download DOTA-v1.0 raw imagery (~2 GB on first run; images up to ~13,000 px per side)
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("DOTAv1.yaml", autodownload=True)
print("train:", data["train"])
print("val:  ", data["val"])

In [ ]:
# Tile into overlapping 1024x1024 crops (single-scale: split_dota's own defaults,
# crop=1024 gap=200 rate=1.0 -- multiscale rates=[0.5,1,1.5] would ~triple both
# preprocessing time and training-set size for a further mAP gain; left as a
# documented option, not the default, to control compute cost)
import yaml
from ultralytics.data.split_dota import split_trainval

DOTA_ROOT  = Path("/content/datasets/DOTAv1")
SPLIT_ROOT = Path("/content/datasets/DOTAv1-split")

if not (SPLIT_ROOT / "images" / "train").exists():
    split_trainval(data_root=str(DOTA_ROOT), save_dir=str(SPLIT_ROOT))

n_train = len(list((SPLIT_ROOT / "images" / "train").glob("*")))
n_val   = len(list((SPLIT_ROOT / "images" / "val").glob("*")))
print(f"tiles: train={n_train}  val={n_val}")

CLASS_NAMES = [
    "plane", "ship", "storage tank", "baseball diamond", "tennis court",
    "basketball court", "ground track field", "harbor", "bridge",
    "large vehicle", "small vehicle", "helicopter", "roundabout",
    "soccer ball field", "swimming pool",
]
data_yaml = {
    "path": str(SPLIT_ROOT), "train": "images/train", "val": "images/val",
    "names": dict(enumerate(CLASS_NAMES)),
}
DOTA_YAML = Path("/content/dota_obb.yaml")
DOTA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(f"wrote {DOTA_YAML}")

## Train (yolo26s-obb, imgsz=1024)

Checkpoints are written to Drive every epoch. **If Colab disconnects**: reconnect,
set `RESUME = True` in the configuration cell, then Runtime → Run all — training
continues from `last.pt`.

This recipe hasn't been timed end-to-end before (unlike the VisDrone 640 run, which
had a known per-epoch cost from the start) — 15,749 train tiles is more images/epoch
than VisDrone's 6,471, and the OBB head has an extra angle-regression branch. Watch
the wall-clock time printed after the first epoch; `patience=20` will stop early if
the model converges before 100 epochs, same as both VisDrone runs did.

In [ ]:
import time
from ultralytics import YOLO

last_pt = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
if RESUME and last_pt.exists():
    print(f"resuming from {last_pt}")
    model = YOLO(str(last_pt))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    t0 = time.time()
    model.train(
        data=str(DOTA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE,
        batch=BATCH, cache=CACHE, seed=SEED, project=str(RUNS_DIR), name=RUN_NAME,
        exist_ok=True,
    )
    print(f"training wall time: {(time.time() - t0) / 3600:.2f} h")

In [ ]:
# Validate the best checkpoint (built-in rotated-IoU OBB mAP -- no separate
# pycocotools pipeline needed here, unlike the VisDrone axis-aligned-box track)
best_pt = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
metrics = YOLO(str(best_pt)).val(data=str(DOTA_YAML), imgsz=IMGSZ, split="val")
print(f"val mAP50={metrics.box.map50:.4f}  mAP50-95={metrics.box.map:.4f}")

In [ ]:
# Copy artifacts to a stable Drive path
import shutil

shutil.copy2(best_pt, WEIGHTS_DIR / "yolo26s_dota_1024.pt")
results_csv = RUNS_DIR / RUN_NAME / "results.csv"
if results_csv.exists():
    shutil.copy2(results_csv, WEIGHTS_DIR / f"results_{RUN_NAME}.csv")
print("artifacts on Drive:")
for p in sorted(WEIGHTS_DIR.iterdir()):
    print(" ", p)
print("\nNext: download yolo26s_dota_1024.pt into the repo's weights/ folder.")